In [5]:
import pandas as pd

df = pd.read_parquet("/home/jovyan/work/data/bronze/university/students.parquet")
print(df.shape)
df.head()

(5000, 10)


,student_id,first_name,last_name,email,birth_date,enrolled_at,country,_source_file,_source_domain,_ingested_at
0,STU-0000001,Martina,Diaz,martina.diaz5727@lake.local,2000-12-21,2019-10-01,US,students.csv,university,2026-07-20T01:48:13.310178+00:00
1,STU-0000002,Manuel,Torres,manuel.torres5619@mail.test,2004-08-10,2025-03-20,CL,students.csv,university,2026-07-20T01:48:13.310178+00:00
2,STU-0000003,Maximiliano,Martinez,maximiliano.martinez7688@demo.io,2007-08-10,2022-10-19,PE,students.csv,university,2026-07-20T01:48:13.310178+00:00
3,STU-0000004,Magdalena,Vasquez,magdalena.vasquez8686@example.com,2002-05-13,2020-09-23,CL,students.csv,university,2026-07-20T01:48:13.310178+00:00
4,STU-0000005,Luis,Rivera,luis.rivera9349@lake.local,2005-12-18,2025-07-28,CL,students.csv,university,2026-07-20T01:48:13.310178+00:00


In [ ]:
import glob

total = 0
for f in glob.glob("/home/jovyan/work/data/bronze/*/*.parquet"):
    n = len(pd.read_parquet(f))
    total += n
print(f"Total filas en Bronze (parquet): {total}")

In [6]:
customers = pd.read_parquet("/home/jovyan/work/data/bronze/billing/customers.parquet")
print(customers["external_ref"].isna().mean())

0.5


In [7]:
customers["has_external_ref"] = customers["external_ref"].notna()

# ¿Se correlaciona con otras columnas? Por ejemplo, segment o country
print(customers.groupby("has_external_ref")["segment"].value_counts(normalize=True))
print()
print(customers.groupby("has_external_ref")["country"].value_counts(normalize=True).head(10))

# ¿Se correlaciona con la fecha de creación? (ej: todos los migrados son "viejos")
print()
print(customers.groupby("has_external_ref")["created_at"].agg(["min", "max"]))

has_external_ref  segment   
False             retail        0.6996
                  smb           0.2234
                  enterprise    0.0770
True              retail        0.7042
                  smb           0.2166
                  enterprise    0.0792
Name: proportion, dtype: float64

has_external_ref  country
False             CL         0.3986
                  AR         0.1022
                  PE         0.0998
                  MX         0.0988
                  ES         0.0816
                  BR         0.0800
                  CO         0.0752
                  US         0.0638
True              CL         0.4120
                  PE         0.0984
Name: proportion, dtype: float64

                                  min                  max
has_external_ref                                          
False             2018-01-01 02:21:56  2025-12-30 13:12:16
True              2018-01-01 16:15:42  2025-12-30 13:31:31


In [9]:
def clean_customers(df):
    df = base_clean(df, date_cols=["created_at"], text_cols=[...], dedup_key=["customer_id"])
    df["has_external_ref"] = df["external_ref"].notna() 
    return df

In [10]:
import pandas as pd

students_silver = pd.read_parquet("/home/jovyan/work/data/silver/university/students.parquet")
print("Estudiantes con edad inválida:", (~students_silver["_valid_age"]).sum(), "de", len(students_silver))

subs_silver = pd.read_parquet("/home/jovyan/work/data/silver/billing/subscriptions.parquet")
print("Suscripciones con rango de fecha inválido:", (~subs_silver["_valid_date_range"]).sum(), "de", len(subs_silver))

customers_silver = pd.read_parquet("/home/jovyan/work/data/silver/billing/customers.parquet")
print("Columnas de customers en Silver:", customers_silver.columns.tolist())

opportunities_silver = pd.read_parquet("/home/jovyan/work/data/silver/crm/opportunities.parquet")
print("dtypes de opportunities:", opportunities_silver.dtypes[["created_at", "close_date", "amount"]].to_dict())

Estudiantes con edad inválida: 636 de 5000
Suscripciones con rango de fecha inválido: 783 de 15000
Columnas de customers en Silver: ['customer_id', 'external_ref', 'first_name', 'last_name', 'email', 'country', 'created_at', 'segment', '_bronze_ingested_at']
dtypes de opportunities: {'created_at': dtype('<M8[ns]'), 'close_date': dtype('<M8[ns]'), 'amount': dtype('float64')}


In [ ]:
import glob

total = 0
for f in glob.glob("/home/jovyan/work/data/bronze/*/*.parquet"):
    n = len(pd.read_parquet(f))
    total += n
print(f"Total filas en Bronze (parquet): {total}")

In [2]:
students = pd.read_parquet("/home/jovyan/work/data/silver/university/students.parquet")
customers = pd.read_parquet("/home/jovyan/work/data/silver/billing/customers.parquet")
contacts = pd.read_parquet("/home/jovyan/work/data/silver/crm/contacts.parquet")

def full_name(df):
    return (df["first_name"].str.strip().str.lower() + " " + df["last_name"].str.strip().str.lower())

students_names = set(full_name(students))
customers_names = set(full_name(customers))
contacts_names = set(full_name(contacts))

print(f"students ↔ customers (por nombre): {len(students_names & customers_names)} de {len(students)}")
print(f"students ↔ contacts (por nombre): {len(students_names & contacts_names)} de {len(students)}")
print(f"customers ↔ contacts (por nombre): {len(customers_names & contacts_names)} de {len(customers)}")

students ↔ customers (por nombre): 2096 de 5000
students ↔ contacts (por nombre): 2140 de 5000
customers ↔ contacts (por nombre): 2436 de 10000


In [3]:
print("Nombres únicos en students:", len(students_names), "de", len(students), "filas")
print("Nombres únicos en customers:", len(customers_names), "de", len(customers), "filas")
print("Nombres únicos en contacts:", len(contacts_names), "de", len(contacts), "filas")

Nombres únicos en students: 2146 de 5000 filas
Nombres únicos en customers: 2441 de 10000 filas
Nombres únicos en contacts: 2494 de 15000 filas


In [4]:
merged = students.merge(customers, left_on=full_name(students), right_on=full_name(customers), suffixes=("_student", "_customer"))
same_email = (merged["email_student"] == merged["email_customer"]).sum()
print(f"De los que coinciden por nombre, cuántos también coinciden por email: {same_email} de {len(merged)}")

De los que coinciden por nombre, cuántos también coinciden por email: 1 de 19865
